# Does the orthophoto add signal that elevation alone can't see?

Topographic detection (notebook 03) plateaus at hundreds of candidates vs. ~30-35 real dolines -- bare rock roughness looks the same as real depressions in elevation alone. Idea: real bowl-shaped depressions (wet or dry) should look locally *darker* than their surroundings in the orthophoto -- shadow from the bowl shape itself, or water pooling. Testing whether that's a real, separable signal before building on it, same way we calibrated DevFromMeanElev against manual points.

In [1]:
import os
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("pyproject.toml not found in any parent directory")


os.chdir(_find_repo_root(Path.cwd()))

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject

from master_thesis.doline_detection import compute_dev_from_mean_elev, random_points_in_polygon, sample_raster

DATE = "20250820"
ORTHO = Path(f"data/raw/{DATE}/{DATE}_terra_ppk_rgb_om.tif")
AOI = Path(f"data/raw/{DATE}/{DATE}_aoi.gpkg")
MANUAL_POINTS = Path("data/manual/Sinkholes.shp")
INTERIM = Path("data/interim")
TARGET_RES_M = 0.05  # match the LiDAR grid so this is combinable with topographic rasters later

In [2]:
aoi = gpd.read_file(AOI)
minx, miny, maxx, maxy = aoi.total_bounds
width = int((maxx - minx) / TARGET_RES_M)
height = int((maxy - miny) / TARGET_RES_M)
dst_transform = rasterio.transform.from_bounds(minx, miny, maxx, maxy, width, height)

brightness_path = INTERIM / f"{DATE}_ortho_brightness_5cm.tif"

with rasterio.open(ORTHO) as src:
    rgb = np.zeros((3, height, width), dtype=np.float32)
    for band in range(3):
        reproject(
            source=rasterio.band(src, band + 1),
            destination=rgb[band],
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs=src.crs,
            resampling=Resampling.average,
        )
    crs = src.crs

brightness = rgb.mean(axis=0)
meta = {
    "driver": "GTiff", "dtype": "float32", "count": 1, "crs": crs,
    "transform": dst_transform, "width": width, "height": height, "nodata": 0,
}
with rasterio.open(brightness_path, "w", **meta) as dst:
    dst.write(brightness[np.newaxis, :, :])
print(f"wrote {brightness_path} ({width}x{height} at {TARGET_RES_M}m)")

wrote data\interim\20250820_ortho_brightness_5cm.tif (30512x11362 at 0.05m)


In [3]:
manual_points = gpd.read_file(MANUAL_POINTS).to_crs(crs)
background = random_points_in_polygon(
    aoi.to_crs(crs).union_all(), n=300, crs=crs,
    exclude=manual_points.geometry, exclude_buffer_m=2.0, seed=0,
)

# same physical scales that worked for topography -- shadow/pooling within a
# multi-metre bowl is a similar-scale phenomenon, no reason to expect otherwise
WINDOW_WIDTHS_M = [2, 4, 6, 10]

records = []
for width_m in WINDOW_WIDTHS_M:
    filter_px = int(round(width_m / TARGET_RES_M)) | 1
    dev_path = INTERIM / f"{DATE}_ortho_brightness_5cm_dev{filter_px}.tif"
    if not dev_path.exists():
        compute_dev_from_mean_elev(brightness_path, dev_path, filter_px)

    manual_vals = sample_raster(dev_path, manual_points)
    background_vals = sample_raster(dev_path, background)
    for val in manual_vals:
        records.append({"window_m": width_m, "group": "manual doline", "brightness_dev": val})
    for val in background_vals:
        records.append({"window_m": width_m, "group": "background", "brightness_dev": val})

results = pd.DataFrame(records)
results.groupby(["window_m", "group"])["brightness_dev"].describe()[["mean", "std", "min", "50%", "max"]]

mean       std       min       50%       max
window_m group                                                          
2        background     0.093504  0.959507 -2.807286  0.093246  2.786864
         manual doline -1.728387  1.191551 -3.502439 -1.951678  1.531506
4        background     0.109856  0.957493 -2.613288  0.100651  3.109514
         manual doline -2.394380  1.084272 -4.502670 -2.628605  0.849058
6        background     0.110411  0.960828 -2.595151  0.090064  3.143595
         manual doline -2.632951  0.983655 -4.522977 -2.850256  0.151263
10       background     0.108844  0.969964 -2.378608  0.089097  3.100231
         manual doline -2.885287  0.867533 -4.555515 -2.895933 -0.507256

In [4]:
THRESHOLDS = np.arange(-60, 0, 1.0)

for w in WINDOW_WIDTHS_M:
    m = results.loc[(results.window_m == w) & (results.group == "manual doline"), "brightness_dev"].to_numpy()
    b = results.loc[(results.window_m == w) & (results.group == "background"), "brightness_dev"].to_numpy()
    recalls = np.array([(m < t).mean() for t in THRESHOLDS])
    fprs = np.array([(b < t).mean() for t in THRESHOLDS])
    j = recalls - fprs
    i = j.argmax()
    print(f"window={w}m  best_threshold={THRESHOLDS[i]:+.1f}  recall={recalls[i]:.0%}  fpr={fprs[i]:.0%}  J={j[i]:.2f}")

window=2m  best_threshold=-1.0  recall=79%  fpr=13%  J=0.67
window=4m  best_threshold=-1.0  recall=91%  fpr=14%  J=0.77
window=6m  best_threshold=-1.0  recall=94%  fpr=14%  J=0.80
window=10m  best_threshold=-2.0  recall=85%  fpr=1%  J=0.84


## Leave-one-out cross-validation

Same rigor as the topographic calibration -- don't trust a fit-on-all-19-points result until it survives holding each point out.

In [5]:
manual_wide = pd.DataFrame(index=range(len(manual_points)))
background_wide = pd.DataFrame(index=range(len(background)))
for width_m in WINDOW_WIDTHS_M:
    filter_px = int(round(width_m / TARGET_RES_M)) | 1
    dev_path = INTERIM / f"{DATE}_ortho_brightness_5cm_dev{filter_px}.tif"
    manual_wide[width_m] = sample_raster(dev_path, manual_points)
    background_wide[width_m] = sample_raster(dev_path, background)


def best_threshold(manual_col, background_col, thresholds):
    recalls = np.array([(manual_col < t).mean() for t in thresholds])
    fprs = np.array([(background_col < t).mean() for t in thresholds])
    j = recalls - fprs
    i = j.argmax()
    return thresholds[i], recalls[i], fprs[i], j[i]


loocv_hits = []
loocv_windows = []
for i in range(len(manual_wide)):
    train = manual_wide.drop(index=i)
    held_out = manual_wide.loc[i]

    best_j, best_w, best_t = -1.0, None, None
    for w in WINDOW_WIDTHS_M:
        t, _, _, j = best_threshold(train[w].to_numpy(), background_wide[w].to_numpy(), THRESHOLDS)
        if j > best_j:
            best_j, best_w, best_t = j, w, t

    loocv_hits.append(held_out[best_w] < best_t)
    loocv_windows.append(best_w)

loocv_recall = np.mean(loocv_hits)
print(f"leave-one-out recall: {loocv_recall:.0%} ({sum(loocv_hits)}/{len(loocv_hits)})")
print("chosen windows:", pd.Series(loocv_windows).value_counts().to_dict())

leave-one-out recall: 85% (29/34)
chosen windows: {10: 34}


## The real test: full-AOI candidate count

Point-level stats looked great for the topographic signal too, right up until it produced 14,920 candidates on the full raster (see notebook 03 / experiment_log.md). Don't trust this until it survives the same test.

In [6]:
from master_thesis.doline_detection import extract_anomalies

best_window_overall = pd.Series(loocv_windows).mode().iloc[0]
final_t, final_recall, final_fpr, final_j = best_threshold(
    manual_wide[best_window_overall].to_numpy(), background_wide[best_window_overall].to_numpy(), THRESHOLDS
)
print(f"using window={best_window_overall}m threshold={final_t:.1f} (recall={final_recall:.0%}, fpr={final_fpr:.0%})")

filter_px = int(round(best_window_overall / TARGET_RES_M)) | 1
dev_path = INTERIM / f"{DATE}_ortho_brightness_5cm_dev{filter_px}.tif"
spectral_candidates = extract_anomalies(dev_path, threshold_m=final_t, min_area_m2=0.5)
print(f"{len(spectral_candidates)} candidates from brightness signal alone, full AOI")

using window=10m threshold=-2.0 (recall=85%, fpr=1%)


198 candidates from brightness signal alone, full AOI
